# Federated Unlearning Based on Memorization

In [ ]:
dataset_dir = "./dataset"
root_path= './'

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [ ]:
CUDA_LAUNCH_BLOCKING=1

In [ ]:
from DL_classification_lib import *
from DL_vision_dataset import *
from DL_vision_dataset_sampling import *
from DL_vision_model import *
from General_utils import *
from Image_utils import *

In [ ]:
from Federated_Learning_lib import *
from DL_check_lib import *

from FL_unlearning_fine_tune_by_tracing_gradient_stat import *
from FL_unlearning_utility import *

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import grad
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
import torchvision.utils as vutils
from torchvision import models, datasets, transforms
from torchvision.models import *

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import random
import re
import copy
import time
import math
import logging

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Dataset

### Cifar10

In [ ]:
transform = transforms.Compose([
                      transforms.ToTensor(),
                      #transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
                      transforms.Resize([224,224])
                    ])

In [ ]:
Cifar10_train_dataset = Cifar10(root=dataset_dir, train=True, transform=transform)
Cifar10_test_dataset = Cifar10(root=dataset_dir, train=False, transform=transform)
Cifar_dataset_size = len(Cifar10_train_dataset)

In [ ]:
len(Cifar10_train_dataset)

In [ ]:
Cifar10_train_loader = DataLoader(Cifar10_train_dataset, batch_size =128, shuffle=False)
Cifar10_test_loader = DataLoader(Cifar10_test_dataset, batch_size =128, shuffle=False)

### Cifar100

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2761)),
])

transform = transforms.Compose([
      transforms.Resize([32,32]),
      transforms.ToTensor(),
      transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2761)),
])

In [ ]:
transform_no_augmentation = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2761)),
])

In [ ]:
Cifar100_train_dataset = Cifar100(root=dataset_dir, train=True, transform=transform_train)
Cifar100_train_dataset_no_augmentation = Cifar100(root=dataset_dir, train=True, transform=transform_no_augmentation)
Cifar100_test_dataset = Cifar100(root=dataset_dir, train=False, transform=transform)
Cifar100_dataset_size = len(Cifar100_train_dataset)

In [ ]:
len(Cifar100_train_dataset)

In [ ]:
Cifar100_train_loader = DataLoader(Cifar100_train_dataset, batch_size = 32, shuffle=True)
Cifar100_test_loader = DataLoader(Cifar100_test_dataset, batch_size = 32, shuffle=False)

### EMNIST

In [ ]:
transform = transforms.Compose([
                      transforms.ToTensor()
                    ])
target_transform = lambda y: y - 1

In [ ]:
EMNIST_train_dataset = torchvision.datasets.EMNIST(root=dataset_dir+"/EMNIST/", split ="letters", train=True, download=True, transform=transform, target_transform= target_transform)
EMNIST_test_dataset = torchvision.datasets.EMNIST(root=dataset_dir+"/EMNIST/", split = "letters", train=False, download=True, transform=transform, target_transform= target_transform)
EMNIST_dataset_size = len(EMNIST_train_dataset)

In [ ]:
len(EMNIST_train_dataset)

In [ ]:
len(np.unique(EMNIST_train_dataset.targets))

In [ ]:
np.unique(EMNIST_train_dataset.targets)

In [ ]:
EMNIST_train_loader = DataLoader(EMNIST_train_dataset, batch_size =128, shuffle=False)
EMNIST_test_loader = DataLoader(EMNIST_test_dataset, batch_size =128, shuffle=False)

## Lib

In [ ]:
def drop_neuron_by_indices(layer, indices):
    with torch.no_grad():
        weight = layer.weight.data

        weight[indices] = 0.

In [ ]:
def get_initialized_weights(shape, method="normal", device="cuda"):
    import torch.nn.init as init
    """Returns a new tensor with initialized values based on the chosen method."""
    new_tensor = torch.empty(shape).to(device)  # Create an empty tensor of the given shape
    if method == "xavier_uniform":
        init.xavier_uniform_(new_tensor)
    elif method == "xavier_normal":
        init.xavier_normal_(new_tensor)
    elif method == "kaiming_uniform":
        init.kaiming_uniform_(new_tensor, nonlinearity='relu')
    elif method == "kaiming_normal":
        init.kaiming_normal_(new_tensor, nonlinearity='relu')
    elif method == "normal":
        init.normal_(new_tensor, mean=0.0, std=0.1)
    elif method == "uniform":
        init.uniform_(new_tensor, a=-0.1, b=0.1)
    else:
        raise ValueError(f"Unknown initialization method: {method}")
    
    return new_tensor

In [ ]:
def drop_neuron_by_indices(model, indices_by_keys):
    with torch.no_grad():
        state_dict = copy.deepcopy(model.state_dict())

        for key in indices_by_keys:
            weight = state_dict[key]
            #print(weight.shape)
            
            indices = indices_by_keys[key]
            #print(indices.shape)
            #print(weight.shape)
            if indices == None:
                continue

            idx = indices.t()
            #weight[tuple(idx)] = 0.
            #print(weight[tuple(idx)].shape)

            if len(weight.shape) > 1:
                initialized_weights = get_initialized_weights(weight.shape, method="kaiming_uniform", device="cuda")
                weight[tuple(idx)] = initialized_weights[tuple(idx)]
            else:
                weight[tuple(idx)] = get_initialized_weights(weight[tuple(idx)].shape, method="uniform", device="cuda")
        
            state_dict[key] = weight
            
    return state_dict

In [ ]:
def reset_model_state_dict(model):
    with torch.no_grad():
        state_dict = copy.deepcopy(model.state_dict())

        for key in state_dict.keys():
            weight = state_dict[key]

            if len(weight.shape) > 1:
                weight = get_initialized_weights(weight.shape, method="kaiming_uniform", device="cuda")
            else:
                weight = get_initialized_weights(weight.shape, method="uniform", device="cuda")
        
            state_dict[key] = weight
            
    return state_dict

In [ ]:
def measure_memorization_scores(within_model,without_models,dataset):

    loader = DataLoader(dataset, batch_size = 128, shuffle = False)

    baseline_examples_prob = epoch_test_probability(loader, within_model, only_label=True)

    exclude_examples_probs = []

    for without_model in without_models:
    
        exclude_examples_prob = epoch_test_probability(loader, without_model, only_label=True)
        exclude_examples_probs.append(exclude_examples_prob)

    memorization_scores = baseline_examples_prob - torch.mean(torch.stack(exclude_examples_probs), dim=0)

    return memorization_scores

## Model Architecture

### Resnet18

In [ ]:
def get_classification_model_generator_with_Resnet18(type_name, num_classes, **kwargs):
    
    model_dict={
        "Original":OriginalResNet18
    }
    
    def generator():

        model = model_dict[type_name](num_classes = num_classes,**kwargs)
        reset_state_dict = reset_model_state_dict(model)
        model.load_state_dict(reset_state_dict)
        
        return model

    return generator



class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1, group_norm_groups=4, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        #self.bn1 = nn.Identity()
        self.bn1 = nn.GroupNorm(group_norm_groups, planes, eps=1e-05, affine=False)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        #self.bn2 = nn.Identity()
        self.bn2 = nn.GroupNorm(group_norm_groups, planes, eps=1e-05, affine=False)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out



#Original Resnet18
class OriginalResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super(OriginalResNet18, self).__init__()
        
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.GroupNorm(4, 64, eps=1e-05, affine=False)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        # self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        # self.bn1 = nn.GroupNorm(4, 64, eps=1e-05, affine=False)
        # self.relu = nn.ReLU(inplace=True)

        #Define the 4 layers as per the structure
        self.layer1 = self._make_layer(64, 64, 2, stride=1, group_norm_groups=4)
        self.layer2 = self._make_layer(64, 128, 2, stride=2, group_norm_groups=8)
        self.layer3 = self._make_layer(128, 256, 2, stride=2, group_norm_groups=16)
        self.layer4 = self._make_layer(256, 512, 2, stride=2, group_norm_groups=32)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_planes, planes, blocks, stride=1, group_norm_groups=4, downsample=None):
        if stride != 1 or in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.GroupNorm(group_norm_groups, planes, eps=1e-05, affine=False),
            )

        layers = []
        layers.append(BasicBlock(in_planes, planes, stride, group_norm_groups, downsample))
        for _ in range(1, blocks):
            layers.append(BasicBlock(planes, planes, group_norm_groups=group_norm_groups))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

### Resnet34

In [ ]:
def get_classification_model_generator_with_Resnet34(type_name, num_classes, **kwargs):

    def generator():
        model = OriginalResNet34(num_classes=num_classes, **kwargs)
        reset_state_dict = reset_model_state_dict(model)
        model.load_state_dict(reset_state_dict)
        return model

    return generator


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, group_norm_groups=4, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.GroupNorm(group_norm_groups, planes, eps=1e-05, affine=False)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.GroupNorm(group_norm_groups, planes, eps=1e-05, affine=False)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

class OriginalResNet34(nn.Module):
    def __init__(self, num_classes=100):
        super(OriginalResNet34, self).__init__()
        self.in_planes = 64

        # self.conv1 = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False)
        # self.bn1 =  nn.GroupNorm(4, 64, eps=1e-05, affine=False)
        # self.relu = nn.ReLU(inplace=True)
        # self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
        
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.GroupNorm(4, 64, eps=1e-05, affine=False)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(64, 64, blocks=3, stride=1, group_norm_groups=4)
        self.layer2 = self._make_layer(64, 128, blocks=4, stride=2, group_norm_groups=8)
        self.layer3 = self._make_layer(128, 256, blocks=6, stride=2, group_norm_groups=16)
        self.layer4 = self._make_layer(256, 512, blocks=3, stride=2, group_norm_groups=32)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_planes, planes, blocks, stride, group_norm_groups):
        downsample = None
        if stride != 1 or in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.GroupNorm(group_norm_groups, planes, eps=1e-05, affine=False),
            )

        layers = [BasicBlock(in_planes, planes, stride, group_norm_groups, downsample)]
        for _ in range(1, blocks):
            layers.append(BasicBlock(planes, planes, group_norm_groups=group_norm_groups))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        # x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


### VGG9

In [ ]:
def get_classification_model_generator_with_VGG9(num_classes, type_name="Original", **kwargs):
    
    model_dict={
        "Original":VGG9_GN
    }
    
    def generator():

        model = model_dict[type_name](num_classes = num_classes,**kwargs)
        reset_state_dict = reset_model_state_dict(model)
        model.load_state_dict(reset_state_dict)
        
        return model

    return generator



class VGG9_GN(nn.Module):
    def __init__(self, num_classes=26, num_groups=8):
        super(VGG9_GN, self).__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups, 64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups, 64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # Output: (64, 14, 14)

            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups, 128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups, 128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # Output: (128, 7, 7)
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 7 * 7, 256),
            nn.ReLU(inplace=True),
            #nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x

# Exp Cifar10 - IID - Basic Experiments

## Dataset Preparation

In [ ]:
client_number = 10
client_dataset_size = 5000

In [ ]:
indices = np.arange(len(Cifar10_train_dataset))
sampling_indices = [np.random.choice(indices, client_dataset_size, replace=False) for _ in range(client_number)]
sampling_name = make_name(["Cifar10","Resnet18","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
pickle_save(sampling_path, sampling_indices)

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

## FedAvg Normally Training

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 3 
epochs = 100
model_generator = get_classification_model_generator_with_Resnet18("Original", 10)
test_data_loader = DataLoader(Cifar10_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar10_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar10_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader=val_data_loader, lr_scheduler=False)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","IID","Normal_Training","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","IID","Normal_Training","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### General Test

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
model = server.global_model

In [ ]:
target_client_index = 0

In [ ]:
train_acc, train_loss = epoch_test(Cifar10_train_loader, model)
print(train_acc, train_loss)

In [ ]:
test_acc, test_loss = epoch_test(Cifar10_test_loader, model)
print(test_acc, test_loss)

In [ ]:
target_client_index = 0
target_client_dataset = SamplerBasedOnIndices(Cifar10_train_dataset, sampling_indices[target_client_index])
target_client_loader = DataLoader(target_client_dataset, batch_size = 128, shuffle = False)
target_acc, target_loss = epoch_test(target_client_loader, model)
print(target_acc, target_loss)

## FedAvg Normally Training - Retraining Baseline

### Model 0

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 3 
epochs = 100
model_generator = get_classification_model_generator_with_Resnet18("Original", 10)
test_data_loader = DataLoader(Cifar10_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar10_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar10_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader=val_data_loader, lr_scheduler=False)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","IID","Normal","Retraining","Index",str(target_client_index),"Model","0"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","IID","Normal","Retraining","Index",str(target_client_index),"Model","0"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### Model 1

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 3 
epochs = 100
model_generator = get_classification_model_generator_with_Resnet18("Original", 10)
test_data_loader = DataLoader(Cifar10_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar10_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar10_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader=val_data_loader, lr_scheduler=False)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","IID","Normal","Retraining","Index",str(target_client_index),"Model","1"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","IID","Normal","Retraining","Index",str(target_client_index),"Model","1"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### Model 2

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 3 
epochs = 100
model_generator = get_classification_model_generator_with_Resnet18("Original", 10)
test_data_loader = DataLoader(Cifar10_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar10_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar10_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader=val_data_loader, lr_scheduler=False)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","IID","Normal","Retraining","Index",str(target_client_index),"Model","2"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","IID","Normal","Retraining","Index",str(target_client_index),"Model","2"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### General Test

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
model = server.global_model

In [ ]:
target_client_index = 0

In [ ]:
train_acc, train_loss = epoch_test(Cifar10_train_loader, model)
print(train_acc, train_loss)

In [ ]:
test_acc, test_loss = epoch_test(Cifar10_test_loader, model)
print(test_acc, test_loss)

In [ ]:
target_client_dataset = SamplerBasedOnIndices(Cifar10_train_dataset, sampling_indices[target_client_index])
target_client_loader = DataLoader(target_client_dataset, batch_size = 128, shuffle = False)
target_acc, target_loss = epoch_test(target_client_loader, model)
print(target_acc, target_loss)

## Memorization Scores

In [ ]:
model_generator = get_classification_model_generator_with_Resnet18("Original", 10)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","IID","Normal_Training","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
within_model= load_model_based_generator(model_generator, model_path)

In [ ]:
without_models = []

for i in range(1):
    model_name = make_name(["Cifar10","Resnet18","IID","Normal","Retraining","Index",str(target_client_index),"Model",str(i)],"")
    model_path = make_path(root_path,"model",model_name)
    without_model= load_model_based_generator(model_generator, model_path)

without_models.append(without_model)

In [ ]:
target_client_index = 0

sampling_name = make_name(["Cifar10","Resnet18","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

target_client_dataset = SamplerBasedOnIndices(Cifar10_train_dataset, sampling_indices[target_client_index])

In [ ]:
memorization_scores = measure_memorization_scores(within_model,without_models,target_client_dataset)
memorization_scores[memorization_scores < 0.] = 0 

In [ ]:
mem_score_name = make_name(["Cifar10","Resnet18","IID","MemScore",str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
pickle_save(mem_score_path, memorization_scores.cpu())

In [ ]:
mem_score_name = make_name(["Cifar10","Resnet18","IID","MemScore", str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
memorization_scores = pickle_load(mem_score_path)

In [ ]:
plt.hist(memorization_scores, bins=20)

In [ ]:
# bound = 85
# threshold = np.percentile(memorization_scores, bound)
# threshold

# Exp Cifar10 - IID - Unlearning Experiments

## Fed Unlearning Normal Training

In [ ]:
client_number = 10
client_dataset_size = 5000
#client_dataset_size = 1
local_epochs = 3 
epochs = 100

retraining_model_generator = get_classification_model_generator_with_Resnet18("Original", 10)

model_generator = get_classification_model_generator_with_Resnet18("Original", 10)

test_data_loader = DataLoader(Cifar10_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4} ) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar10_train_dataset, sampling_index) for sampling_index in sampling_indices]

In [ ]:
log_name = make_name(["Cifar10","Resnet18","IID","Based_Fine_Tune","Normal_Training"],".log")
log_path = make_path(root_path,"log",log_name)
logger = get_logger(log_path, name=log_name, verbosity=1)

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar10_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, 
    batch_size = 128, 
    val_data_loader = val_data_loader, 
    lr_scheduler=False, 
    run_local_dataset = True)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

## Fed NL Pruning & Fine-tuning

In [ ]:
drop_ratio = 0.3
pruning_epoches = [0]

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 3 
epochs = 30

model_generator = get_classification_model_generator_with_Resnet18("Original", 10)
test_data_loader = DataLoader(Cifar10_test_dataset, batch_size = 128, shuffle = False)

target_client_index = 0

In [ ]:
repeat_round = 0

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}, optim.lr_scheduler.MultiStepLR, {'milestones':[1], 'gamma':0.2}) for _ in range(client_number)]

In [ ]:
model_name = make_name(["Cifar10","Resnet18","IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar10_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar10_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed, epoch_data = fedAvg_epoch_fine_tuning_while_pruning_redundant_neurons(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, 
    val_data_loader=val_data_loader, 
    lr_scheduler=False, 
    target_client_index=target_client_index, 
    pruning_epoches=pruning_epoches,
    return_dynamic = True,
    drop_ratio=drop_ratio,
    round_drop_ratio=1,
    round_random_drop=False)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
save_model(server.global_model, model_path)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
unlearning_model = load_model(server.global_model, model_path)

### General Test

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
model = load_model_based_generator(model_generator, model_path)

In [ ]:
target_client_index = 0

In [ ]:
# train_acc, train_loss = epoch_test(MNIST_train_loader, model)
# print(train_acc, train_loss)

In [ ]:
test_acc, test_loss = epoch_test(Cifar10_test_loader, model)
print(test_acc, test_loss)

In [ ]:
target_client_dataset = SamplerBasedOnIndices(Cifar10_train_dataset, sampling_indices[target_client_index])
target_client_loader = DataLoader(target_client_dataset, batch_size = 128, shuffle = False)
target_acc, target_loss = epoch_test(target_client_loader, model)
print(target_acc, target_loss)

# Exp: Cifar10 - Non-IID （Dirichlet）- Basic Experiments

## Dataset Preparation

In [ ]:
client_number = 10
client_dataset_size = 5000

In [ ]:
dirichlet_dist = np.random.dirichlet([0.5] * client_number, 10)
np.sum(dirichlet_dist, axis=1)

In [ ]:
# class_num = 10
# alpha = 0.5

# labels = np.array(Cifar10_train_dataset.labels)

# dirichlet_dist = np.random.dirichlet([alpha] * client_number, class_num)

# sampling_indices = [[] for _ in range(client_number)]
# for c in range(class_num):  # For each class
#     indices = np.where(labels == c)[0]
#     np.random.shuffle(indices)
#     class_split = np.split(indices, (dirichlet_dist[c] * len(indices)).astype(int).cumsum())[:-1]
#     for i in range(client_number):
#         sampling_indices[i].extend(class_split[i])

# sampling_name = make_name(["Cifar10","Resnet18","Non-IID","Sampling","Indices"],"")
# sampling_path = make_path(root_path,"data",sampling_name)
# pickle_save(sampling_path, sampling_indices)

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

## FedAvg Normally Training

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 3 
epochs = 100
model_generator = get_classification_model_generator_with_Resnet18("Original", 10)
test_data_loader = DataLoader(Cifar10_test_dataset, batch_size = 128, shuffle = False)

class_num = 10
each_class_num = 5000
alpha = 0.5

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar10_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar10_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader = val_data_loader, lr_scheduler=False, run_local_dataset = True)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

## FedAvg Normally Training - Retraining Baseline

### Model 0

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 3 
epochs = 100
model_generator = get_classification_model_generator_with_Resnet18("Original", 10)
test_data_loader = DataLoader(Cifar10_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
# server = Server(model_generator)
# clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.8},  optim.lr_scheduler.MultiStepLR, {'milestones':[25,50], 'gamma':0.1}) for _ in range(client_number)]

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar10_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar10_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader=val_data_loader, 
    lr_scheduler=False)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","0"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","0"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### Model 1

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 3 
epochs = 100
model_generator = get_classification_model_generator_with_Resnet18("Original", 10)
test_data_loader = DataLoader(Cifar10_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
# server = Server(model_generator)
# clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.8},  optim.lr_scheduler.MultiStepLR, {'milestones':[25,50], 'gamma':0.1}) for _ in range(client_number)]

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar10_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar10_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader=val_data_loader, 
    lr_scheduler=False)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","1"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","1"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### Model 2

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 3 
epochs = 100
model_generator = get_classification_model_generator_with_Resnet18("Original", 10)
test_data_loader = DataLoader(Cifar10_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
# server = Server(model_generator)
# clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.8},  optim.lr_scheduler.MultiStepLR, {'milestones':[25,50], 'gamma':0.1}) for _ in range(client_number)]

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar10_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar10_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader=val_data_loader, 
    lr_scheduler=False)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","2"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","2"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### General Test

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","2"],"")
model_path = make_path(root_path,"model",model_name)
model = load_model_based_generator(model_generator, model_path)

In [ ]:
train_acc, train_loss = epoch_test(Cifar10_train_loader, model)
print(train_acc, train_loss)

In [ ]:
test_acc, test_loss = epoch_test(Cifar10_test_loader, model)
print(test_acc, test_loss)

In [ ]:
target_client_dataset = SamplerBasedOnIndices(Cifar10_train_dataset, sampling_indices[target_client_index])
target_client_loader = DataLoader(target_client_dataset, batch_size = 128, shuffle = False)
target_acc, target_loss = epoch_test(target_client_loader, model)
print(target_acc, target_loss)

## Memorization Scores

In [ ]:
model_generator = get_classification_model_generator_with_Resnet18("Original", 10)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Normal","Training","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
within_model= load_model_based_generator(model_generator, model_path)

In [ ]:
without_models = []

for i in range(3):
    model_name = make_name(["Cifar10","Resnet18","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model",str(i)],"")
    model_path = make_path(root_path,"model",model_name)
    without_model= load_model_based_generator(model_generator, model_path)

without_models.append(without_model)

In [ ]:
target_client_index = 0

sampling_name = make_name(["Cifar10","Resnet18","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

target_client_dataset = SamplerBasedOnIndices(Cifar10_train_dataset, sampling_indices[target_client_index])

In [ ]:
memorization_scores = measure_memorization_scores(within_model,without_models,target_client_dataset)
memorization_scores[memorization_scores < 0.] = 0 

In [ ]:
mem_score_name = make_name(["Cifar10","Resnet18","Non-IID","MemScore",str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
pickle_save(mem_score_path, memorization_scores.cpu())

In [ ]:
mem_score_name = make_name(["Cifar10","Resnet18","Non-IID","MemScore", str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
memorization_scores = pickle_load(mem_score_path)

In [ ]:
plt.hist(memorization_scores, bins=20)

# Exp Cifar10 - Non-IID - Unlearning Experiments

## Fed Unlearning Normal Training

In [ ]:
client_number = 10
client_dataset_size = 5000
#client_dataset_size = 1
local_epochs = 3 
epochs = 50

retraining_model_generator = get_classification_model_generator_with_Resnet18("Original", 10)

model_generator = get_classification_model_generator_with_Resnet18("Original", 10)

test_data_loader = DataLoader(Cifar10_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4} ) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar10_train_dataset, sampling_index) for sampling_index in sampling_indices]

In [ ]:
log_name = make_name(["Cifar10","Resnet18","Non-IID","Based_Fine_Tune","Normal_Training"],".log")
log_path = make_path(root_path,"log",log_name)
logger = get_logger(log_path, name=log_name, verbosity=1)

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar10_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, 
    batch_size = 128, 
    val_data_loader = val_data_loader, 
    lr_scheduler=False, 
    run_local_dataset = True)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

## Fed NL Pruning & Fine-tuning

In [ ]:
drop_ratio = 0.4
pruning_epoches = [0]

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 3 
epochs = 30
model_generator = get_classification_model_generator_with_Resnet18("Original", 10)
test_data_loader = DataLoader(Cifar10_test_dataset, batch_size = 128, shuffle = False)

target_client_index = 0

In [ ]:
repeat_round = 0

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
# server.set_aggregate_rule(aggregate_parameter_with_momentum_sa(server, client_lr = 1e-4, max_lr = 2e-4, step_size = int(epochs/2), momentum = 0.3))

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar10_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar10_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed, epoch_data = fedAvg_epoch_fine_tuning_while_pruning_redundant_neurons(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, 
    val_data_loader=val_data_loader, 
    lr_scheduler=False, 
    target_client_index=target_client_index, 
    pruning_epoches=pruning_epoches,
    return_dynamic = True,
    drop_ratio=drop_ratio,
    round_drop_ratio=1,
    round_random_drop=True)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
save_model(server.global_model, model_path)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
unlearning_model = load_model(server.global_model, model_path)

### General Test

In [ ]:
sampling_name = make_name(["Cifar10","Resnet18","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
model_name = make_name(["Cifar10","Resnet18","Non-IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
model = load_model_based_generator(model_generator, model_path)

In [ ]:
target_client_index = 0

In [ ]:
# train_acc, train_loss = epoch_test(MNIST_train_loader, model)
# print(train_acc, train_loss)

In [ ]:
test_acc, test_loss = epoch_test(Cifar10_test_loader, model)
print(test_acc, test_loss)

In [ ]:
target_client_dataset = SamplerBasedOnIndices(Cifar10_train_dataset, sampling_indices[target_client_index])
target_client_loader = DataLoader(target_client_dataset, batch_size = 128, shuffle = False)
target_acc, target_loss = epoch_test(target_client_loader, model)
print(target_acc, target_loss)

# Exp Cifar100 - IID - Basic Experiments

## Dataset Preparation

In [ ]:
client_number = 10
client_dataset_size = 5000

In [ ]:
# indices = np.arange(len(Cifar100_train_dataset))
# sampling_indices = [np.random.choice(indices, client_dataset_size, replace=False) for _ in range(client_number)]
# sampling_name = make_name(["Cifar100","Resnet34","IID","Sampling","Indices"],"")
# sampling_path = make_path(root_path,"data",sampling_name)
# pickle_save(sampling_path, sampling_indices)

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

## FedAvg Normally Training

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 10
epochs = 100
model_generator = get_classification_model_generator_with_Resnet34("Original", 100)
test_data_loader = DataLoader(Cifar100_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.9, 'weight_decay':5e-4}, optim.lr_scheduler.CosineAnnealingLR, {'T_max':epochs*local_epochs}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar100_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar100_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 64, val_data_loader=val_data_loader, lr_scheduler=True)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","IID","Normal_Training","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","IID","Normal_Training","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### General Test

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
model = server.global_model

In [ ]:
target_client_index = 0

In [ ]:
train_acc, train_loss = epoch_test(Cifar100_train_loader, model)
print(train_acc, train_loss)

In [ ]:
test_acc, test_loss = epoch_test(Cifar100_test_loader, model)
print(test_acc, test_loss)

In [ ]:
target_client_index = 0
target_client_dataset = SamplerBasedOnIndices(Cifar100_train_dataset, sampling_indices[target_client_index])
target_client_loader = DataLoader(target_client_dataset, batch_size = 128, shuffle = False)
target_acc, target_loss = epoch_test(target_client_loader, model)
print(target_acc, target_loss)

## FedAvg Normally Training - Retraining Baseline

### Model 0

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 10 
epochs = 100
model_generator = get_classification_model_generator_with_Resnet34("Original", 100)
test_data_loader = DataLoader(Cifar100_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.9, 'weight_decay':5e-4}, optim.lr_scheduler.CosineAnnealingLR, {'T_max':epochs*local_epochs}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar100_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar100_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 64, val_data_loader=val_data_loader, lr_scheduler=True)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","IID","Normal","Retraining","Index",str(target_client_index),"Model","0"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","IID","Normal","Retraining","Index",str(target_client_index),"Model","0"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### Model 1

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 10 
epochs = 100
model_generator = get_classification_model_generator_with_Resnet34("Original", 100)
test_data_loader = DataLoader(Cifar100_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.9, 'weight_decay':5e-4}, optim.lr_scheduler.CosineAnnealingLR, {'T_max':epochs*local_epochs}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar100_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar100_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 64, val_data_loader=val_data_loader, lr_scheduler=True)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","IID","Normal","Retraining","Index",str(target_client_index),"Model","1"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","IID","Normal","Retraining","Index",str(target_client_index),"Model","1"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### Model 2

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 10 
epochs = 100
model_generator = get_classification_model_generator_with_Resnet34("Original", 100)
test_data_loader = DataLoader(Cifar100_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.9, 'weight_decay':5e-4}, optim.lr_scheduler.CosineAnnealingLR, {'T_max':epochs*local_epochs}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar100_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar100_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 64, val_data_loader=val_data_loader, lr_scheduler=True)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","IID","Normal","Retraining","Index",str(target_client_index),"Model","2"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","IID","Normal","Retraining","Index",str(target_client_index),"Model","2"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### General Test

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
model = server.global_model

In [ ]:
target_client_index = 0

In [ ]:
train_acc, train_loss = epoch_test(Cifar100_train_loader, model)
print(train_acc, train_loss)

In [ ]:
test_acc, test_loss = epoch_test(Cifar100_test_loader, model)
print(test_acc, test_loss)

In [ ]:
target_client_dataset = SamplerBasedOnIndices(Cifar100_train_dataset, sampling_indices[target_client_index])
target_client_loader = DataLoader(target_client_dataset, batch_size = 128, shuffle = False)
target_acc, target_loss = epoch_test(target_client_loader, model)
print(target_acc, target_loss)

## Memorization Scores

In [ ]:
model_generator = get_classification_model_generator_with_Resnet34("Original", 100)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","IID","Normal_Training","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
within_model= load_model_based_generator(model_generator, model_path)

In [ ]:
without_models = []

for i in range(3):
    model_name = make_name(["Cifar100","Resnet34","IID","Normal","Retraining","Index",str(target_client_index),"Model",str(i)],"")
    model_path = make_path(root_path,"model",model_name)
    without_model= load_model_based_generator(model_generator, model_path)

without_models.append(without_model)

In [ ]:
target_client_index = 0

sampling_name = make_name(["Cifar100","Resnet34","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

target_client_dataset = SamplerBasedOnIndices(Cifar100_train_dataset, sampling_indices[target_client_index])

In [ ]:
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar100_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
memorization_scores = measure_memorization_scores(within_model,without_models,target_client_dataset)
memorization_scores[memorization_scores < 0.] = 0 

In [ ]:
mem_score_name = make_name(["Cifar100","Resnet34","IID","MemScore",str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
pickle_save(mem_score_path, memorization_scores.cpu())

In [ ]:
mem_score_name = make_name(["Cifar100","Resnet34","IID","MemScore", str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
memorization_scores = pickle_load(mem_score_path)

In [ ]:
plt.hist(memorization_scores, bins=20)

In [ ]:
bound = 85
threshold = np.percentile(memorization_scores, bound)
threshold

# Exp Cifar100 - IID - Unlearning Experiments

## Fed Unlearning Normal Training

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 10 
epochs = 100

retraining_model_generator = get_classification_model_generator_with_Resnet34("Original", 100)

model_generator = get_classification_model_generator_with_Resnet34("Original", 100)

test_data_loader = DataLoader(Cifar100_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.9, 'weight_decay':5e-4}, optim.lr_scheduler.CosineAnnealingLR, {'T_max':epochs*local_epochs}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar100_train_dataset, sampling_index) for sampling_index in sampling_indices]

In [ ]:
log_name = make_name(["Cifar100","Resnet34","IID","Based_Fine_Tune","Normal_Training"],".log")
log_path = make_path(root_path,"log",log_name)
logger = get_logger(log_path, name=log_name, verbosity=1)

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar100_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, 
    batch_size = 64, 
    val_data_loader = val_data_loader, 
    lr_scheduler=False, 
    run_local_dataset = True)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

## Fed NL Pruning & Fine-tuning

In [ ]:
drop_ratio = 0.8
pruning_epoches = [0]

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 10 
epochs = 40
model_generator = get_classification_model_generator_with_Resnet34("Original", 100)
test_data_loader = DataLoader(Cifar100_test_dataset, batch_size = 128, shuffle = False)

target_client_index = 0

In [ ]:
repeat_round = 0

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.9, 'weight_decay':5e-4}, optim.lr_scheduler.CosineAnnealingLR, {'T_max':epochs*local_epochs}) for _ in range(client_number)]

In [ ]:
model_name = make_name(["Cifar100","Resnet34","IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar100_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar100_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed, epoch_data = fedAvg_epoch_fine_tuning_while_pruning_redundant_neurons(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 64, 
    val_data_loader=val_data_loader, 
    lr_scheduler=True, 
    target_client_index=target_client_index, 
    pruning_epoches=pruning_epoches,
    return_dynamic = True,
    drop_ratio=drop_ratio,
    round_drop_ratio=1,
    round_random_drop=True)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
save_model(server.global_model, model_path)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
unlearning_model = load_model(server.global_model, model_path)

### General Test

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
model = load_model_based_generator(model_generator, model_path)

In [ ]:
target_client_index = 0

In [ ]:
# train_acc, train_loss = epoch_test(MNIST_train_loader, model)
# print(train_acc, train_loss)

In [ ]:
test_acc, test_loss = epoch_test(Cifar100_test_loader, model)
print(test_acc, test_loss)

In [ ]:
target_client_dataset = SamplerBasedOnIndices(Cifar100_train_dataset, sampling_indices[target_client_index])
target_client_loader = DataLoader(target_client_dataset, batch_size = 128, shuffle = False)
target_acc, target_loss = epoch_test(target_client_loader, model)
print(target_acc, target_loss)

# Exp: Cifar100 - Non-IID （Dirichlet）- Basic Experiments

## Dataset Preparation

In [ ]:
client_number = 10
client_dataset_size = 5000

In [ ]:
dirichlet_dist = np.random.dirichlet([0.5] * client_number, 10)
np.sum(dirichlet_dist, axis=1)

In [ ]:
# class_num = 100
# alpha = 0.5

# labels = np.array(Cifar100_train_dataset.labels)

# dirichlet_dist = np.random.dirichlet([alpha] * client_number, class_num)

# sampling_indices = [[] for _ in range(client_number)]
# for c in range(class_num):  # For each class
#     indices = np.where(labels == c)[0]
#     np.random.shuffle(indices)
#     class_split = np.split(indices, (dirichlet_dist[c] * len(indices)).astype(int).cumsum())[:-1]
#     for i in range(client_number):
#         sampling_indices[i].extend(class_split[i])

# sampling_name = make_name(["Cifar100","Resnet34","Non-IID","Sampling","Indices"],"")
# sampling_path = make_path(root_path,"data",sampling_name)
# pickle_save(sampling_path, sampling_indices)

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
for sampling_index in sampling_indices:
    print(len(sampling_index))

In [ ]:
len(Cifar100_train_dataset)

## FedAvg Normally Training

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 10 
epochs = 100
model_generator = get_classification_model_generator_with_Resnet34("Original", 100)
test_data_loader = DataLoader(Cifar100_test_dataset, batch_size = 128, shuffle = False)

class_num = 100
each_class_num = 5000
alpha = 0.5

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.9, 'weight_decay':5e-4}, optim.lr_scheduler.CosineAnnealingLR, {'T_max':epochs*local_epochs}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar100_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar100_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 64, 
    val_data_loader = val_data_loader, 
    lr_scheduler=True, 
    run_local_dataset = True)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

## FedAvg Normally Training - Retraining Baseline

### Model 0

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 10 
epochs = 100
model_generator = get_classification_model_generator_with_Resnet34("Original", 100)
test_data_loader = DataLoader(Cifar100_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
# server = Server(model_generator)
# clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.8},  optim.lr_scheduler.MultiStepLR, {'milestones':[25,50], 'gamma':0.1}) for _ in range(client_number)]

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.9, 'weight_decay':5e-4}, optim.lr_scheduler.CosineAnnealingLR, {'T_max':epochs*local_epochs}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar100_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar100_train_dataset_no_augmentation,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 64, val_data_loader=val_data_loader, 
    lr_scheduler=True)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","0"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","0"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### Model 1

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 10
epochs = 100
model_generator = get_classification_model_generator_with_Resnet34("Original", 100)
test_data_loader = DataLoader(Cifar100_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
# server = Server(model_generator)
# clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.8},  optim.lr_scheduler.MultiStepLR, {'milestones':[25,50], 'gamma':0.1}) for _ in range(client_number)]

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.9, 'weight_decay':5e-4}, optim.lr_scheduler.CosineAnnealingLR, {'T_max':epochs*local_epochs}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar100_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar100_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader=val_data_loader, 
    lr_scheduler=True)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","1"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","1"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### Model 2

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 10 
epochs = 100
model_generator = get_classification_model_generator_with_Resnet34("Original", 100)
test_data_loader = DataLoader(Cifar100_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
# server = Server(model_generator)
# clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.8},  optim.lr_scheduler.MultiStepLR, {'milestones':[25,50], 'gamma':0.1}) for _ in range(client_number)]

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.9, 'weight_decay':5e-4}, optim.lr_scheduler.CosineAnnealingLR, {'T_max':epochs*local_epochs}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar100_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar100_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 64, val_data_loader=val_data_loader, 
    lr_scheduler=True)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","2"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","2"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### General Test

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","2"],"")
model_path = make_path(root_path,"model",model_name)
model = load_model_based_generator(model_generator, model_path)

In [ ]:
train_acc, train_loss = epoch_test(Cifar100_train_loader, model)
print(train_acc, train_loss)

In [ ]:
test_acc, test_loss = epoch_test(Cifar100_test_loader, model)
print(test_acc, test_loss)

In [ ]:
target_client_dataset = SamplerBasedOnIndices(Cifar100_train_dataset, sampling_indices[target_client_index])
target_client_loader = DataLoader(target_client_dataset, batch_size = 128, shuffle = False)
target_acc, target_loss = epoch_test(target_client_loader, model)
print(target_acc, target_loss)

## Memorization Scores

In [ ]:
model_generator = get_classification_model_generator_with_Resnet34("Original", 100)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
within_model= load_model_based_generator(model_generator, model_path)

In [ ]:
without_models = []

for i in range(3):
    model_name = make_name(["Cifar100","Resnet34","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model",str(i)],"")
    model_path = make_path(root_path,"model",model_name)
    without_model= load_model_based_generator(model_generator, model_path)

without_models.append(without_model)

In [ ]:
target_client_index = 0

sampling_name = make_name(["Cifar100","Resnet34","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

target_client_dataset = SamplerBasedOnIndices(Cifar100_train_dataset, sampling_indices[target_client_index])

In [ ]:
memorization_scores = measure_memorization_scores(within_model,without_models,target_client_dataset)
memorization_scores[memorization_scores < 0.] = 0 

In [ ]:
mem_score_name = make_name(["Cifar100","Resnet34","Non-IID","MemScore",str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
pickle_save(mem_score_path, memorization_scores.cpu())

In [ ]:
mem_score_name = make_name(["Cifar100","Resnet34","Non-IID","MemScore", str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
memorization_scores = pickle_load(mem_score_path)

In [ ]:
plt.hist(memorization_scores, bins=20)

# Exp Cifar100 - Non-IID - Unlearning Experiments

## Fed Unlearning Normal Training

In [ ]:
client_number = 100
client_dataset_size = 5000
#client_dataset_size = 1
local_epochs = 10 
epochs = 100

retraining_model_generator = get_classification_model_generator_with_Resnet34("Original", 100)

model_generator = get_classification_model_generator_with_Resnet34("Original", 100)

test_data_loader = DataLoader(Cifar100_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.9, 'weight_decay':5e-4}, optim.lr_scheduler.CosineAnnealingLR, {'T_max':epochs*local_epochs}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar100_train_dataset, sampling_index) for sampling_index in sampling_indices]

In [ ]:
log_name = make_name(["Cifar100","Resnet34","Non-IID","Based_Fine_Tune","Normal_Training"],".log")
log_path = make_path(root_path,"log",log_name)
logger = get_logger(log_path, name=log_name, verbosity=1)

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar100_train_dataset_no_augmentation,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, 
    batch_size = 64, 
    val_data_loader = val_data_loader, 
    lr_scheduler=False, 
    run_local_dataset = True)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

## Fed NL Pruning & Fine-tuning

In [ ]:
drop_ratio = 0.85
pruning_epoches = [0]

In [ ]:
client_number = 10
client_dataset_size = 5000
local_epochs = 5 
epochs = 40
model_generator = get_classification_model_generator_with_Resnet34("Original", 100)
test_data_loader = DataLoader(Cifar100_test_dataset, batch_size = 128, shuffle = False)

target_client_index = 0

In [ ]:
repeat_round = 0

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.SGD, {'lr':1e-2,'momentum':0.9,'weight_decay':5e-4}, optim.lr_scheduler.CosineAnnealingLR, {'T_max':epochs*local_epochs}) for _ in range(client_number)]

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(Cifar100_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(Cifar100_train_dataset_no_augmentation,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed, epoch_data = fedAvg_epoch_fine_tuning_while_pruning_redundant_neurons(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 64, 
    val_data_loader=val_data_loader, 
    lr_scheduler=True, 
    target_client_index=target_client_index, 
    pruning_epoches=pruning_epoches,
    return_dynamic = True,
    drop_ratio=drop_ratio,
    round_drop_ratio=1,
    round_random_drop=True)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
save_model(server.global_model, model_path)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
unlearning_model = load_model(server.global_model, model_path)

### General Test

In [ ]:
sampling_name = make_name(["Cifar100","Resnet34","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
model_name = make_name(["Cifar100","Resnet34","Non-IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
model = load_model_based_generator(model_generator, model_path)

In [ ]:
target_client_index = 0

In [ ]:
# train_acc, train_loss = epoch_test(MNIST_train_loader, model)
# print(train_acc, train_loss)

In [ ]:
test_acc, test_loss = epoch_test(Cifar100_test_loader, model)
print(test_acc, test_loss)

In [ ]:
target_client_dataset = SamplerBasedOnIndices(Cifar100_train_dataset, sampling_indices[target_client_index])
target_client_loader = DataLoader(target_client_dataset, batch_size = 128, shuffle = False)
target_acc, target_loss = epoch_test(target_client_loader, model)
print(target_acc, target_loss)

In [ ]:
transform = transforms.Compose([
                      transforms.ToTensor()
                    ])
target_transform = lambda y: y - 1

In [ ]:
EMNIST_train_dataset = torchvision.datasets.EMNIST(root="./dataset/EMNIST/", split ="letters", train=True, download=True, transform=transform, target_transform= target_transform)
EMNIST_test_dataset = torchvision.datasets.EMNIST(root="./dataset/EMNIST/", split = "letters", train=False, download=True, transform=transform, target_transform= target_transform)
EMNIST_dataset_size = len(EMNIST_train_dataset)

In [ ]:
len(EMNIST_train_dataset)

In [ ]:
len(np.unique(EMNIST_train_dataset.targets))

In [ ]:
np.unique(EMNIST_train_dataset.targets)

In [ ]:
EMNIST_train_loader = DataLoader(EMNIST_train_dataset, batch_size =128, shuffle=False)
EMNIST_test_loader = DataLoader(EMNIST_test_dataset, batch_size =128, shuffle=False)

# Exp: EMNIST - IID - Basic Experiments

## Dataset Preparation

In [ ]:
client_number = 10
client_dataset_size = 12000

In [ ]:
# indices = np.arange(len(EMNIST_train_dataset))
# sampling_indices = [np.random.choice(indices, client_dataset_size, replace=False) for _ in range(client_number)]
# sampling_name = make_name(["EMNIST","VGG9","IID","Sampling","Indices"],"")
# sampling_path = make_path(root_path,"data",sampling_name)
# pickle_save(sampling_path, sampling_indices)

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
print([len(indices) for indices in sampling_indices])

## FedAvg Normally Training

In [ ]:
client_number = 10
class_num = 26
client_dataset_size = 12000
local_epochs = 3
epochs = 100
model_generator = get_classification_model_generator_with_VGG9(class_num)
test_data_loader = DataLoader(EMNIST_test_dataset, batch_size = 128, shuffle = False)

# each_class_num = 5000
# alpha = 0.5

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(EMNIST_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(EMNIST_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader = val_data_loader, lr_scheduler=False, run_local_dataset = True)

In [ ]:
model_name = make_name(["EMNIST","VGG9","IID","Normal","Training","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["EMNIST","VGG9","IID","Normal","Training","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

## FedAvg Normally Training - Retraining Baseline

### Model 0

In [ ]:
client_number = 10
client_dataset_size = 12000
local_epochs = 3 
epochs = 100
model_generator = get_classification_model_generator_with_VGG9(26)
test_data_loader = DataLoader(EMNIST_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
# server = Server(model_generator)
# clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.8},  optim.lr_scheduler.MultiStepLR, {'milestones':[25,50], 'gamma':0.1}) for _ in range(client_number)]

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(EMNIST_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(EMNIST_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader=val_data_loader, 
    lr_scheduler=False)

In [ ]:
model_name = make_name(["EMNIST","VGG9","IID","Normal","Retraining","Index",str(target_client_index),"Model","0"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["EMNIST","VGG9","IID","Normal","Retraining","Index",str(target_client_index),"Model","0"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### Model 1

In [ ]:
client_number = 10
client_dataset_size = 12000
local_epochs = 3 
epochs = 100
model_generator = get_classification_model_generator_with_VGG9(26)
test_data_loader = DataLoader(EMNIST_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
# server = Server(model_generator)
# clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.8},  optim.lr_scheduler.MultiStepLR, {'milestones':[25,50], 'gamma':0.1}) for _ in range(client_number)]

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(EMNIST_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(EMNIST_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader=val_data_loader, 
    lr_scheduler=False)

In [ ]:
model_name = make_name(["EMNIST","VGG9","IID","Normal","Retraining","Index",str(target_client_index),"Model","1"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["EMNIST","VGG9","IID","Normal","Retraining","Index",str(target_client_index),"Model","1"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### Model 2

In [ ]:
client_number = 10
client_dataset_size = 12000
local_epochs = 3 
epochs = 100
model_generator = get_classification_model_generator_with_VGG9(26)
test_data_loader = DataLoader(EMNIST_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
# server = Server(model_generator)
# clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.8},  optim.lr_scheduler.MultiStepLR, {'milestones':[25,50], 'gamma':0.1}) for _ in range(client_number)]

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(EMNIST_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(EMNIST_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader=val_data_loader, 
    lr_scheduler=False)

In [ ]:
model_name = make_name(["EMNIST","VGG9","IID","Normal","Retraining","Index",str(target_client_index),"Model","2"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["EMNIST","VGG9","IID","Normal","Retraining","Index",str(target_client_index),"Model","2"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

## Memorization Scores

In [ ]:
model_generator = get_classification_model_generator_with_VGG9(26)

In [ ]:
model_name = make_name(["EMNIST","VGG9","IID","Normal","Training","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
within_model= load_model_based_generator(model_generator, model_path)

In [ ]:
without_models = []

for i in range(3):
    model_name = make_name(["EMNIST","VGG9","IID","Normal","Retraining","Index",str(target_client_index),"Model",str(i)],"")
    model_path = make_path(root_path,"model",model_name)
    without_model= load_model_based_generator(model_generator, model_path)

without_models.append(without_model)

In [ ]:
target_client_index = 0

sampling_name = make_name(["EMNIST","VGG9","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

target_client_dataset = SamplerBasedOnIndices(EMNIST_train_dataset, sampling_indices[target_client_index])

In [ ]:
memorization_scores = measure_memorization_scores(within_model,without_models,target_client_dataset)
memorization_scores[memorization_scores < 0.] = 0 

In [ ]:
mem_score_name = make_name(["EMNIST","VGG9","IID","MemScore",str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
pickle_save(mem_score_path, memorization_scores.cpu())

In [ ]:
mem_score_name = make_name(["EMNIST","VGG9","IID","MemScore", str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
memorization_scores = pickle_load(mem_score_path)

In [ ]:
plt.hist(memorization_scores, bins=20)

# Exp EMNIST - IID - Unlearning Experiments

## Fed Unlearning Normal Training

In [ ]:
client_number = 26
client_dataset_size = 12000
#client_dataset_size = 1
local_epochs = 3 
epochs = 100

retraining_model_generator = get_classification_model_generator_with_VGG9(26)

model_generator = get_classification_model_generator_with_VGG9(26)

test_data_loader = DataLoader(EMNIST_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4} ) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(EMNIST_train_dataset, sampling_index) for sampling_index in sampling_indices]

In [ ]:
log_name = make_name(["EMNIST","VGG9","IID","Based_Fine_Tune","Normal_Training"],".log")
log_path = make_path(root_path,"log",log_name)
logger = get_logger(log_path, name=log_name, verbosity=1)

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(EMNIST_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, 
    batch_size = 128, 
    val_data_loader = val_data_loader, 
    lr_scheduler=False, 
    run_local_dataset = True)

In [ ]:
model_name = make_name(["EMNIST","VGG9","IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["EMNIST","VGG9","IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

## Fed NL Pruning & Fine-tuning

In [ ]:
drop_ratio = 0.3
pruning_epoches = [0]

In [ ]:
client_number = 10
client_dataset_size = 12000
local_epochs = 3 
epochs = 30
model_generator = get_classification_model_generator_with_VGG9(26)
test_data_loader = DataLoader(EMNIST_test_dataset, batch_size = 128, shuffle = False)

target_client_index = 0

In [ ]:
repeat_round = 0

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}, optim.lr_scheduler.MultiStepLR, {'milestones':[1], 'gamma':0.2}) for _ in range(client_number)]

In [ ]:
model_name = make_name(["EMNIST","VGG9","IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(EMNIST_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(EMNIST_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed, epoch_data = fedAvg_epoch_fine_tuning_while_pruning_redundant_neurons(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, 
    val_data_loader=val_data_loader, 
    lr_scheduler=False, 
    target_client_index=target_client_index, 
    pruning_epoches=pruning_epoches,
    return_dynamic = True,
    drop_ratio=drop_ratio,
    round_drop_ratio=1,
    round_random_drop=True)

In [ ]:
model_name = make_name(["EMNIST","VGG9","IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
save_model(server.global_model, model_path)

In [ ]:
model_name = make_name(["EMNIST","VGG9","IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
unlearning_model = load_model(server.global_model, model_path)

### General Test

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
model_name = make_name(["EMNIST","VGG9","IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
model = load_model_based_generator(model_generator, model_path)

In [ ]:
target_client_index = 0

In [ ]:
# train_acc, train_loss = epoch_test(MNIST_train_loader, model)
# print(train_acc, train_loss)

In [ ]:
test_acc, test_loss = epoch_test(EMNIST_test_loader, model)
print(test_acc, test_loss)

In [ ]:
target_client_dataset = SamplerBasedOnIndices(EMNIST_train_dataset, sampling_indices[target_client_index])
target_client_loader = DataLoader(target_client_dataset, batch_size = 128, shuffle = False)
target_acc, target_loss = epoch_test(target_client_loader, model)
print(target_acc, target_loss)

# Exp: EMNIST - Non-IID （Dirichlet）- Basic Experiments

## Dataset Preparation

In [ ]:
client_number = 10
client_dataset_size = 12000

In [ ]:
dirichlet_dist = np.random.dirichlet([0.5] * client_number, 26)
np.sum(dirichlet_dist, axis=1)

In [ ]:
# class_num = 26
# alpha = 0.5

# labels = np.array(EMNIST_train_dataset.targets)

# dirichlet_dist = np.random.dirichlet([alpha] * client_number, class_num)

# sampling_indices = [[] for _ in range(client_number)]
# for c in range(class_num):  # For each class
#     indices = np.where(labels == c)[0]
#     np.random.shuffle(indices)
#     class_split = np.split(indices, (dirichlet_dist[c] * len(indices)).astype(int).cumsum())[:-1]
#     for i in range(client_number):
#         sampling_indices[i].extend(class_split[i])

# sampling_name = make_name(["EMNIST","VGG9","Non-IID","Sampling","Indices"],"")
# sampling_path = make_path(root_path,"data",sampling_name)
# pickle_save(sampling_path, sampling_indices)

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
print([len(indices) for indices in sampling_indices])

## FedAvg Normally Training

In [ ]:
client_number = 10
class_num = 26
client_dataset_size = 12000
local_epochs = 3
epochs = 100
model_generator = get_classification_model_generator_with_VGG9(class_num)
test_data_loader = DataLoader(EMNIST_test_dataset, batch_size = 128, shuffle = False)

# each_class_num = 5000
# alpha = 0.5

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(EMNIST_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(EMNIST_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader = val_data_loader, lr_scheduler=False, run_local_dataset = True)

In [ ]:
model_name = make_name(["EMNIST","VGG9","Non-IID","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["EMNIST","VGG9","Non-IID","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

## FedAvg Normally Training - Retraining Baseline

### Model 0

In [ ]:
client_number = 10
client_dataset_size = 12000
local_epochs = 3 
epochs = 100
model_generator = get_classification_model_generator_with_VGG9(26)
test_data_loader = DataLoader(EMNIST_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
# server = Server(model_generator)
# clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.8},  optim.lr_scheduler.MultiStepLR, {'milestones':[25,50], 'gamma':0.1}) for _ in range(client_number)]

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(EMNIST_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(EMNIST_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader=val_data_loader, 
    lr_scheduler=False)

In [ ]:
model_name = make_name(["EMNIST","VGG9","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","0"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["EMNIST","VGG9","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","0"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### Model 1

In [ ]:
client_number = 10
client_dataset_size = 12000
local_epochs = 3 
epochs = 100
model_generator = get_classification_model_generator_with_VGG9(26)
test_data_loader = DataLoader(EMNIST_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
# server = Server(model_generator)
# clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.8},  optim.lr_scheduler.MultiStepLR, {'milestones':[25,50], 'gamma':0.1}) for _ in range(client_number)]

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(EMNIST_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(EMNIST_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader=val_data_loader, 
    lr_scheduler=False)

In [ ]:
model_name = make_name(["EMNIST","VGG9","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","1"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["EMNIST","VGG9","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","1"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

### Model 2

In [ ]:
client_number = 10
client_dataset_size = 12000
local_epochs = 3 
epochs = 100
model_generator = get_classification_model_generator_with_VGG9(26)
test_data_loader = DataLoader(EMNIST_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
# server = Server(model_generator)
# clients = [Client(model_generator, optim.SGD, {'lr':1e-2, 'momentum':0.8},  optim.lr_scheduler.MultiStepLR, {'milestones':[25,50], 'gamma':0.1}) for _ in range(client_number)]

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(EMNIST_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(EMNIST_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, val_data_loader=val_data_loader, 
    lr_scheduler=False)

In [ ]:
model_name = make_name(["EMNIST","VGG9","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","2"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["EMNIST","VGG9","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model","2"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

## Memorization Scores

In [ ]:
model_generator = get_classification_model_generator_with_VGG9(26)

In [ ]:
model_name = make_name(["EMNIST","VGG9","Non-IID","Baseline"],"")
model_path = make_path(root_path,"model",model_name)
within_model= load_model_based_generator(model_generator, model_path)

In [ ]:
without_models = []

for i in range(3):
    model_name = make_name(["EMNIST","VGG9","Non-IID","Normal","Retraining","Index",str(target_client_index),"Model",str(i)],"")
    model_path = make_path(root_path,"model",model_name)
    without_model= load_model_based_generator(model_generator, model_path)

without_models.append(without_model)

In [ ]:
target_client_index = 0

sampling_name = make_name(["EMNIST","VGG9","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

target_client_dataset = SamplerBasedOnIndices(EMNIST_train_dataset, sampling_indices[target_client_index])

In [ ]:
memorization_scores = measure_memorization_scores(within_model,without_models,target_client_dataset)
memorization_scores[memorization_scores < 0.] = 0 

In [ ]:
mem_score_name = make_name(["EMNIST","VGG9","Non-IID","MemScore",str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
pickle_save(mem_score_path, memorization_scores.cpu())

In [ ]:
mem_score_name = make_name(["EMNIST","VGG9","Non-IID","MemScore", str(target_client_index)],"")
mem_score_path = make_path(root_path,"data",mem_score_name)
memorization_scores = pickle_load(mem_score_path)

In [ ]:
plt.hist(memorization_scores, bins=20)

# Exp EMNIST - Non-IID - Unlearning Experiments

## Fed Unlearning Normal Training

In [ ]:
client_number = 26
client_dataset_size = 12000
#client_dataset_size = 1
local_epochs = 3 
epochs = 100

retraining_model_generator = get_classification_model_generator_with_VGG9(26)

model_generator = get_classification_model_generator_with_VGG9(26)

test_data_loader = DataLoader(EMNIST_test_dataset, batch_size = 128, shuffle = False)

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4} ) for _ in range(client_number)]

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(EMNIST_train_dataset, sampling_index) for sampling_index in sampling_indices]

In [ ]:
log_name = make_name(["EMNIST","VGG9","Non-IID","Based_Fine_Tune","Normal_Training"],".log")
log_path = make_path(root_path,"log",log_name)
logger = get_logger(log_path, name=log_name, verbosity=1)

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(EMNIST_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed = fedAvg_epoch(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, 
    batch_size = 128, 
    val_data_loader = val_data_loader, 
    lr_scheduler=False, 
    run_local_dataset = True)

In [ ]:
model_name = make_name(["EMNIST","VGG9","Non-IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.save_model(model_path)

In [ ]:
model_name = make_name(["EMNIST","VGG9","Non-IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

## Fed NL Pruning & Fine-tuning

In [ ]:
drop_ratio = 0.4
pruning_epoches = [0]

In [ ]:
client_number = 10
client_dataset_size = 12000
local_epochs = 3 
epochs = 30
model_generator = get_classification_model_generator_with_VGG9(26)
test_data_loader = DataLoader(EMNIST_test_dataset, batch_size = 128, shuffle = False)

target_client_index = 0

In [ ]:
repeat_round = 0

In [ ]:
server = Server(model_generator)
clients = [Client(model_generator, optim.Adam, {'lr':1e-4}, optim.lr_scheduler.MultiStepLR, {'milestones':[1], 'gamma':0.2}) for _ in range(client_number)]

In [ ]:
model_name = make_name(["EMNIST","VGG9","Non-IID","Based_Fine_Tune","Normal_Training"],"")
model_path = make_path(root_path,"model",model_name)
server.load_model(model_path)

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
client_datasets = [SamplerBasedOnIndices(EMNIST_train_dataset,sampling_index) for sampling_index in sampling_indices]

In [ ]:
target_client_index = 0
val_data_loader = DataLoader(SamplerBasedOnIndices(EMNIST_train_dataset,sampling_indices[target_client_index]), batch_size = 128, shuffle = False)

In [ ]:
clients = clients[0:target_client_index]+clients[target_client_index+1:]
client_datasets = client_datasets[0:target_client_index]+client_datasets[target_client_index+1:]

In [ ]:
len(clients)

In [ ]:
train_acc, train_loss, test_acc, test_loss, time_elapsed, epoch_data = fedAvg_epoch_fine_tuning_while_pruning_redundant_neurons(
    server, clients, client_datasets, test_data_loader, epochs, local_epochs, batch_size = 128, 
    val_data_loader=val_data_loader, 
    lr_scheduler=False, 
    target_client_index=target_client_index, 
    pruning_epoches=pruning_epoches,
    return_dynamic = True,
    drop_ratio=drop_ratio,
    round_drop_ratio=1,
    round_random_drop=True)

In [ ]:
model_name = make_name(["EMNIST","VGG9","Non-IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
save_model(server.global_model, model_path)

In [ ]:
model_name = make_name(["EMNIST","VGG9","Non-IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
unlearning_model = load_model(server.global_model, model_path)

### General Test

In [ ]:
sampling_name = make_name(["EMNIST","VGG9","Non-IID","Sampling","Indices"],"")
sampling_path = make_path(root_path,"data",sampling_name)
sampling_indices = pickle_load(sampling_path)

In [ ]:
model_name = make_name(["EMNIST","VGG9","Non-IID","Based_Fine_Tune_While_Pruning","Index",str(target_client_index),"Model",str(repeat_round)],"")
model_path = make_path(root_path,"model",model_name)
model = load_model_based_generator(model_generator, model_path)

In [ ]:
target_client_index = 0

In [ ]:
# train_acc, train_loss = epoch_test(MNIST_train_loader, model)
# print(train_acc, train_loss)

In [ ]:
test_acc, test_loss = epoch_test(EMNIST_test_loader, model)
print(test_acc, test_loss)

In [ ]:
target_client_dataset = SamplerBasedOnIndices(EMNIST_train_dataset, sampling_indices[target_client_index])
target_client_loader = DataLoader(target_client_dataset, batch_size = 128, shuffle = False)
target_acc, target_loss = epoch_test(target_client_loader, model)
print(target_acc, target_loss)